# **Importing Libraries**

Jalankan cell import libraries, restart kernel , lalu run all

In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio openpyxl
!pip install torch transformers
!pip install git+https://github.com/suno-ai/bark.git -q
!pip install scipy -q
!pip install pydub
!pip install -q transformers==4.57.3
!pip install -q accelerate==1.12.0
!pip install -q bitsandbytes==0.49.1
!pip install -q langchain==1.2.3
!pip install -q langchain-community==0.4.1
!pip install -q langchain-core==1.6.6
!pip install -q langchain-text-splitters
!pip install -q pymupdf
!pip install -q langchain-huggingface
!pip install -q chromadb==1.4.1
!pip install -q huggingface-hub==0.36.0
!pip install -q rank-bm25
!pip install torch
!pip install fastapi-sso
!pip install -q pypdf

In [ ]:
# API & Backend
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio
import threading
import uvicorn
from pyngrok import ngrok
import base64
import subprocess
import tempfile
from pathlib import Path
from fastapi import File, HTTPException, UploadFile, Form
from fastapi.responses import Response
import csv
from openpyxl import load_workbook
from pypdf import PdfReader
import logging
import subprocess
import time
import requests
from kaggle_secrets import UserSecretsClient

In [ ]:
# Models and Processing
import torch
import torchaudio
import soundfile as sf
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from IPython.display import Audio, display
import ipywidgets as widgets
from pydub import AudioSegment
import os
import io
from bark import SAMPLE_RATE, generate_audio, preload_models
from scipy.io.wavfile import write as write_wav
import numpy as np
from fastapi.responses import FileResponse
import nltk
import pandas as pd
nltk.download('punkt_tab')

In [ ]:
# RAG
from langchain_community.document_loaders import WebBaseLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever, EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.retrievers import BM25Retriever
from langchain_core.stores import InMemoryByteStore
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# **File Input Extractor**

In [ ]:
MAX_FILE_SIZE = 10 * 1024 * 1024
SUPPORTED_EXTENSIONS = {".csv", ".docx", ".pdf", ".txt", ".xlsx"}

async def extract_attachment_context(files: list[UploadFile]) -> str:
    if len(files) > 5:
        raise HTTPException(status_code=413, detail="Maksimal 5 dokumen.")

    sections = []
    total_size = 0
    for file in files:
        filename = (file.filename or "").replace("\\", "/").rsplit("/", 1)[-1]
        suffix = Path(filename).suffix.lower()
        if not filename or suffix not in SUPPORTED_EXTENSIONS:
            raise HTTPException(
                status_code=415,
                detail="Format dokumen yang didukung: PDF, DOCX, TXT, CSV, dan XLSX.",
            )

        content = await file.read(MAX_FILE_SIZE + 1)
        if not content:
            raise HTTPException(
                status_code=422,
                detail=f"Dokumen {filename} kosong.",
            )
        if len(content) > MAX_FILE_SIZE:
            raise HTTPException(
                status_code=413,
                detail=f"Ukuran dokumen {filename} melebihi batas 10 MB.",
            )
        total_size += len(content)
        if total_size > 25 * 1024 * 1024:
            raise HTTPException(status_code=413, detail="Total dokumen melebihi 25 MB.")

        if suffix == ".pdf":
            text = "\n".join(
                page.extract_text() or ""
                for page in PdfReader(io.BytesIO(content)).pages
            )
        elif suffix == ".docx":
            document = Document(io.BytesIO(content))
            text = "\n".join(paragraph.text for paragraph in document.paragraphs)
        elif suffix == ".csv":
            try:
                csv_text = content.decode("utf-8-sig")
            except UnicodeDecodeError as exc:
                raise HTTPException(
                    status_code=422,
                    detail=f"Dokumen {filename} bukan CSV UTF-8 yang valid.",
                ) from exc
            rows = csv.reader(io.StringIO(csv_text))
            text = "\n".join(" | ".join(row) for row in rows)
        elif suffix == ".xlsx":
            workbook = load_workbook(
                io.BytesIO(content),
                read_only=True,
                data_only=True,
            )
            try:
                sheets = []
                for worksheet in workbook.worksheets:
                    rows = (
                        " | ".join("" if value is None else str(value) for value in row)
                        for row in worksheet.iter_rows(values_only=True)
                    )
                    sheet_text = "\n".join(rows)
                    sheets.append(f"Sheet: {worksheet.title}\n{sheet_text}")
                text = "\n\n".join(sheets)
            finally:
                workbook.close()
        else:
            try:
                text = content.decode("utf-8-sig")
            except UnicodeDecodeError as exc:
                raise HTTPException(
                    status_code=422,
                    detail=f"Dokumen {filename} bukan teks UTF-8 yang valid.",
                ) from exc

        if not text.strip():
            raise HTTPException(
                status_code=422,
                detail=f"Tidak ada teks yang dapat diekstrak dari {filename}.",
            )
        sections.append(f"--- Dokumen: {filename} ---\n{text}")

    return "\n\n".join(sections)

# **Model Pipeline**

## Model Initialization

In [ ]:
# ============================================================
# 1. SETUP ALOKASI GPU & LOAD MODEL
# ============================================================
device_stt_tts = "cuda:0" if torch.cuda.is_available() else "cpu"
device_llm = "cuda:1" if torch.cuda.device_count() > 1 else device_stt_tts

print(f"STT & TTS berjalan di: {device_stt_tts}")
print(f"LLM Main Engine berjalan di: {device_llm}")

# --- A. Load STT Model (Whisper Large v3 Turbo) ---
print("\n[1/3] Loading STT Model (Whisper Large v3 Turbo)...")
stt_pipeline = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    dtype=torch.float16,
    device=device_stt_tts,
)

# --- B. Load LLM Engine (Qwen2.5-3B-Instruct) ---
print("[2/3] Loading LLM Engine ...")
llm_model_id = "unsloth/Llama-3.2-3B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    dtype=torch.float16,
    device_map=device_llm,
)

# --- C. Load TTS Model (bark) ---
default_load = torch.load

# Fungsi bypass agar model bark bisa dimuat oleh Pytorch versi 2.6
def safe_load_bypass(*args, **kwargs):
    # Memaksa weights_only=False agar model Bark (numpy types) bisa dibaca
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return default_load(*args, **kwargs)

# Terapkan patch
torch.load = safe_load_bypass

preload_models()

print("\n>>> Semua Model Berhasil Dimuat ke GPU VRAM! <<<\n")

## Speech-to-Speech Mode

In [ ]:
def generate_long_audio(text, history_prompt="v2/en_speaker_6", silence_secs=0.25):
    # Pecah teks menjadi kalimat
    sentences = nltk.sent_tokenize(text)
    full_audio = []
    silence = np.zeros(int(silence_secs * SAMPLE_RATE)) # Membuat array hening

    print(f"Total kalimat: {len(sentences)}")

    # Loop per kalimat
    for i, sentence in enumerate(sentences):
        print(f"Sedang memproses kalimat {i+1}")

        # Generate audio per kalimat
        audio_array = generate_audio(sentence, history_prompt=history_prompt)

        # Gabungkan hasil generate
        full_audio.append(audio_array)
        full_audio.append(silence) # Tambah jeda biar natural

    # Ratakan array menjadi satu dimensi
    return np.concatenate(full_audio)

In [ ]:
def sts_mode(audio_input_path: str | Path, speaker_mode: str = "v2/en_speaker_6", files=None):
    # Transformers ASR expects a NumPy array/tensor (or its documented input
    # dictionary), not a pathlib.Path.
    audio_array, audio_sample_rate = sf.read(
        str(audio_input_path),
        dtype="float32",
    )
    if audio_array.size == 0:
        raise ValueError("Audio input WAV kosong.")

    stt_result = stt_pipeline(
        {
            "raw": audio_array,
            "sampling_rate": audio_sample_rate,
        }
    )
    user_text = stt_result["text"].strip()

    if files:
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a friendly voice assistant. Answer concisely and clearly\n"
                    "Answer in the same language as the user's prompt (example: english prompt should be answered in english\n"
                    "without markdown so the response is suitable for text-to-speech.\n"
                    "Give relevant answer based on given files from user\n"
                ),
            },
            {"role": "user", "content": user_text + f"these are the documents: \n{files}"},
        ]
    else:
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a friendly voice assistant. Answer concisely and clearly\n"
                    "without markdown so the response is suitable for text-to-speech\n."
                    ""
                ),
            },
            {"role": "user", "content": user_text},
        ]
    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(device_llm)
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    bot_text = llm_tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    audio_array = generate_long_audio(text=bot_text, history_prompt=speaker_mode)
    if isinstance(audio_array, torch.Tensor):
        audio_array = audio_array.detach().cpu().numpy()
    audio_array = np.asarray(audio_array, dtype=np.float32)

    # Return the actual samples, not IPython.display.Audio. Audio() is for
    # notebook display and does not provide a WAV file path.
    return {
        "user_text": user_text,
        "bot_text": bot_text,
        "audio_array": audio_array,
        "sample_rate": SAMPLE_RATE,
    }

## Text-to-Text Mode

In [ ]:
def chat_mode(chat_prompt, files=None):
    if files:
        messages = [
            {
                "role": "system",
                "content": ("You are a friendly chatbot assistant, answer user question in a friendly manner\n"
                            "Answer in the same language as the user's prompt (example: english prompt should be answered in english\n"
                            "Answer the question based on given files"
                )
            },
            {"role": "user", "content": chat_prompt + f" these are the documents: \n{files}"}
        ]
    else:
        messages = [
            {
                "role": "system",
                "content": "You are a friendly chatbot assistant, answer user question in a friendly manner, provide clear and relevant response based on user's question"
            },
            {"role": "user", "content": chat_prompt}
        ]

    prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(device_llm)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    bot_text = llm_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    return bot_text

## RAG Mode

In [ ]:
# Text-to-text Pipeline
text_generation_pipeline = pipeline(
    model=llm_model,
    tokenizer=llm_tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=500,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

In [ ]:
# Model Initialization
## Embbedding
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda:0'}
)

## Re-Ranker
rerank_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

compressor = CrossEncoderReranker(model=rerank_model, top_n=5)

In [ ]:
def ingest_data(file, embedding_model):
    # Web Loader
    loader = PyMuPDFLoader(file)
    docs = loader.load()

    # Chunker
    ## Parent Chunk
    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
    ## Child Chunk
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

    # Storing
    vectorstore = Chroma(
        collection_name="split_parents",
        embedding_function=embedding_model
    )

    # Store untuk Parent
    docstore = InMemoryByteStore()

    # Assemble Parent-child Document Retriever
    retriever = ParentDocumentRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
        search_type="similarity",
        search_kwargs={"k": 10}
    )

    # Masukkan Data
    retriever.add_documents(docs)

    return retriever, docs

def retrieval(retriever, compressor, docs):
    # Keyword retriever
    bm25_retriever = BM25Retriever.from_documents(docs)
    bm25_retriever.k = 10

    hybrid_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, retriever],
        weights=[0.5, 0.5]
    )

    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=hybrid_retriever,
    )

    return compression_retriever

def generation(llm, compression_retriever, query):
    # Prompt Template
    template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

    You are a friendly assistant,answer user question in a friendly manner.
    Answer the question based on given context. The context is your ground truth
    if you can't find the answer in the context then say that you don't know.
    And also dont hallucinate or give answer outside the context.
    You should differentiate between user's language and document's language.
    if the user asks in english, the answer in english and so on.
    lastly, don't add makrdown symbols in your response.

    Context:
    {context}<|eot_id|><|start_header_id|>user<|end_header_id|>

    {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
    """

    prompt = PromptTemplate(
        template=template,
        input_variables=["context", "question"]
    )

    rag_chain = (
        {"context": compression_retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    response = rag_chain.invoke(query)

    reference_sources = "\n\nSource:\n\n"

    docs = compression_retriever.invoke(query)
    for i, doc in enumerate(docs):
        reference_sources += f"\n{i+1}. Page {doc.metadata.get('page', '?')}"

    return response + reference_sources

## Test Model

### Speech-to-Speech

In [ ]:
uploader = widgets.FileUpload(multiple=False)
display(uploader)

In [ ]:
# Ambil data file yang di-upload
for file_info in uploader.value:
    filename = file_info['name']
    content = file_info['content']
    save_path = os.path.join("/kaggle/working", filename)

    with open(save_path, "wb") as f:
        f.write(content)

    print(f"File berhasil disimpan di: {save_path}")

In [ ]:
# ============================================================
# 3. UJI COBA MENJALANKAN PIPELINE
# ============================================================
# pydub secara otomatis mendeteksi format file
audio = AudioSegment.from_file("Recording (2).m4a")

# 2. Export ke format WAV
audio.export("user_voice_eng.wav", format="wav")
print("Konversi ke WAV selesai!")

In [ ]:
result = sts_mode("/content/user_voice_eng.wav")

# Untuk mendengarkan audio balasan langsung di Kaggle Cell:

display(result["audio_player"])

### Read and Summarize Document Test

In [ ]:
def load_table_as_text(file_path: str) -> str:
    """Membaca file CSV atau Excel dan mengonversinya ke format Markdown Table.

    Format Markdown sangat disukai LLM karena mempertahankan hirarki visual
    baris & kolom.
    """
    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path)
    elif file_path.endswith((".xlsx", ".xls")):
        df = pd.read_excel(file_path)
    else:
        raise ValueError(
            "Format file tidak didukung. Gunakan file .csv atau .xlsx"
        )

    # Mengonversi DataFrame ke string Markdown Table
    try:
        return df.to_markdown(index=False)
    except ImportError:
        # Fallback ke format string CSV jika pustaka 'tabulate' tidak terpasang
        return df.to_csv(index=False)


# 1. Baca data dari file lokal
file_path = "/content/laporan_pendapatan_perusahaan.csv"  # Ganti dengan nama/lokasi file Anda
table_context = load_table_as_text(file_path)

# 3. Susun Prompt (Menyisipkan tabel langsung ke dalam percakapan)
messages = [
    {
        "role": "system",
        "content": (
            "Anda adalah analis data keuangan perusahaan profesional. Tugas Anda adalah membaca "
            "Pastikan anda membaca dokumen secara keseluruhan sebelum melakukan perhitungan (jangan hanya 1 baris)"
            "Misalnya jika ditanya pendapatan bulan januari, hitung semua data (baris) pada bulan tersebut"
            "data tabel yang diberikan, secara otomatis memahami struktur kolom dan definisinya "
            "(seperti kolom pendapatan, tanggal, atau kategori), lalu memberikan jawaban yang singkat dan padat"
            "serta analisis yang akurat."
        ),
    },
    {
        "role": "user",
        "content": f"""Berikut adalah data laporan keuangan:

{table_context}

Berdasarkan data tabel di atas, tolong analisis dan rangkum hal-hal berikut:
1. Berapa total keuntungan bersih pada bulan januari 2025?
2. Kategori produk mana yang memberikan kontribusi terbesar?
3. Sebutkan tren atau insight penting yang Anda temukan dari data tersebut.""",
    },
]

# 4. Terapkan Chat Template Qwen
prompt = llm_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# 5. Jalankan Inferensi
inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)

with torch.no_grad():
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,  # Temperature rendah agar ekstraksi data dan hitungan lebih akurat
        top_p=0.9,
    )

# 6. Ambil Hasil Respons
response_text = llm_tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
)

print("=== HASIL ANALISIS DIREK (DIRECT INGESTION) ===")
print(response_text)

### RAG test

In [ ]:
file_path = "/content/Pedoman Etika Penggunaan AI Akademik.pdf"

retriever, docs = ingest_data(file_path, embedding_model)

compression_retriever = retrieval(retriever, compressor, docs)

In [ ]:
query = "Bagaimana cara kerja sistem RAG menggunakan langchain"

response = generation(llm, compression_retriever, query)

print(response)

# **API Setup**

## Creating FastAPI

In [ ]:
app = FastAPI()

class QueryRequest(BaseModel):
    query: str

@app.get("/health")
def health():
    return {
        "status": "ok",
        "service": "colab"
    }

## Running API

In [ ]:
nest_asyncio.apply()

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

thread = threading.Thread(
    target=run_server,
    daemon=True
)

thread.start()

## Creating Tunnel to Frontend (Via NGROK)

In [ ]:
ngrok.kill()

NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")

ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000)

print(public_url)

# **Model Endpoint**

## Text-to-Text Endpoint

In [ ]:
@app.post("/chat")
async def chat(
    message: str = Form(""),
    files: list[UploadFile] = File(default=[]),
):
    if not message.strip() and not files:
        raise HTTPException(
            status_code=422,
            detail="Pesan atau setidaknya satu dokumen harus disertakan.",
        )

    attachment_context = await extract_attachment_context(files)
    response = chat_mode(message, attachment_context)

    return {"response": response}

## Speech-to-Speech Endpoint

In [ ]:
@app.post("/speech")
async def speech(
    audio: UploadFile = File(...),
    speaker_mode: str = Form("v2/en_speaker_6"),
    files: list[UploadFile] = File(default=[]),
):
    speaker_mode = speaker_mode.strip()

    if not speaker_mode:
        raise HTTPException(
            status_code=422,
            detail="speaker_mode tidak boleh kosong.",
        )
    if not audio.filename:
        raise HTTPException(status_code=422, detail="File audio harus memiliki nama.")

    suffix = Path(audio.filename).suffix.lower() or ".webm"
    if suffix not in {".webm", ".ogg", ".mp4", ".wav"}:
        raise HTTPException(status_code=415, detail="Format audio tidak didukung.")

    audio_content = await audio.read()
    if not audio_content:
        raise HTTPException(status_code=422, detail="File audio kosong.")

    # Ekstrak file sebelum inference; helper ini membaca PDF/DOCX/TXT/CSV/XLSX.
    attachment_context = await extract_attachment_context(files)

    with tempfile.TemporaryDirectory() as temp_dir:
        source_path = Path(temp_dir) / f"input{suffix}"
        wav_path = Path(temp_dir) / "input.wav"
        source_path.write_bytes(audio_content)

        conversion = subprocess.run(
            [
                "ffmpeg", "-y", "-i", str(source_path),
                "-vn", "-ac", "1", "-ar", "16000",
                "-c:a", "pcm_s16le", str(wav_path),
            ],
            capture_output=True,
            text=True,
        )
        if conversion.returncode != 0:
            raise HTTPException(
                status_code=400,
                detail=f"Audio tidak dapat dikonversi ke WAV: {conversion.stderr[-1000:]}",
            )

        result = sts_mode(
            wav_path,
            speaker_mode=speaker_mode,
            files=attachment_context,
        )

        wav_buffer = io.BytesIO()
        sf.write(
            wav_buffer,
            result["audio_array"],
            result["sample_rate"],
            format="WAV",
            subtype="PCM_16",
        )
        bot_text = base64.b64encode(
            result["bot_text"].encode("utf-8")
        ).decode("ascii")
        return Response(
            content=wav_buffer.getvalue(),
            media_type="audio/wav",
            headers={"X-Bot-Text-B64": bot_text},
        )

## RAG Endpoint

### Storing

In [ ]:
@app.post("/store")
async def store_to_vector(file: UploadFile = File(...)):
    # 1. Simpan file sementara agar memiliki file_path
    with tempfile.NamedTemporaryFile(delete=False, suffix=os.path.splitext(file.filename)[1]) as tmp:
        content = await file.read()
        tmp.write(content)
        temp_file_path = tmp.name

    try:
        # 2. Proses ingest & buat retriever
        retriever, docs = ingest_data(temp_file_path, embedding_model)
        compression_retriever = retrieval(retriever, compressor, docs)

        # 3. Simpan retriever ke dalam state aplikasi FastAPI
        app.state.compression_retriever = compression_retriever

        return {"status": "success", "message": f"File {file.filename} berhasil diindeks."}

    finally:
        # Hapus file sementara setelah diproses
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)

### Retrieval & Generation

In [ ]:
@app.post("/rag_text")
def rag_text(payload: QueryRequest):
    # 1. Cek apakah retriever sudah tersedia di app.state
    if not hasattr(app.state, "compression_retriever") or app.state.compression_retriever is None:
        raise HTTPException(
            status_code=400,
            detail="File belum diunggah. Silakan panggil /store terlebih dahulu."
        )

    # 2. Ambil retriever dari state dan jalankan RAG
    retriever = app.state.compression_retriever
    response = generation(llm, retriever, payload.query)

    return response

In [ ]:
@app.post("/rag_speech")
async def rag_speech(audio: UploadFile = File(...)):
    if not audio.filename:
        raise HTTPException(status_code=422, detail="File audio harus memiliki nama.")

    suffix = Path(audio.filename).suffix.lower() or ".webm"
    if suffix not in {".webm", ".ogg", ".mp4", ".wav"}:
        raise HTTPException(status_code=415, detail="Format audio tidak didukung.")

    audio_content = await audio.read()
    if not audio_content:
        raise HTTPException(status_code=422, detail="File audio kosong.")

    with tempfile.TemporaryDirectory() as temp_dir:
        source_path = Path(temp_dir) / f"input{suffix}"
        wav_path = Path(temp_dir) / "input.wav"
        source_path.write_bytes(audio_content)

        conversion = subprocess.run(
            [
                "ffmpeg", "-y", "-i", str(source_path),
                "-vn", "-ac", "1", "-ar", "16000",
                "-c:a", "pcm_s16le", str(wav_path),
            ],
            capture_output=True,
            text=True,
        )
        if conversion.returncode != 0:
            raise HTTPException(
                status_code=400,
                detail=f"Audio tidak dapat dikonversi ke WAV: {conversion.stderr[-1000:]}",
            )

        audio_array, audio_sample_rate = sf.read(
            str(wav_path),
            dtype="float32",
        )
        if audio_array.size == 0:
            raise ValueError("Audio input WAV kosong.")

        stt_result = stt_pipeline(
            {
                "raw": audio_array,
                "sampling_rate": audio_sample_rate,
            }
        )
        user_text = stt_result["text"].strip()

        # 1. Cek apakah retriever sudah tersedia di app.state
        if not hasattr(app.state, "compression_retriever") or app.state.compression_retriever is None:
            raise HTTPException(
                status_code=400,
                detail="File belum diunggah. Silakan panggil /store terlebih dahulu."
            )

        # 2. Ambil retriever dari state dan jalankan RAG
        retriever = app.state.compression_retriever
        response = generation(llm, retriever, user_text)

        audio_array = generate_long_audio(response)
        if isinstance(audio_array, torch.Tensor):
            audio_array = audio_array.detach().cpu().numpy()
        audio_array = np.asarray(audio_array, dtype=np.float32)

        wav_buffer = io.BytesIO()
        sf.write(
            wav_buffer,
            audio_array,
            SAMPLE_RATE,
            format="WAV",
            subtype="PCM_16",
        )
        bot_text = base64.b64encode(
            response.encode("utf-8")
        ).decode("ascii")
        return Response(
            content=wav_buffer.getvalue(),
            media_type="audio/wav",
            headers={"X-Bot-Text-B64": bot_text},
        )